# Agentic Root-Cause Analysis with Investigation Graphs

This notebook demonstrates a multi-agent autonomous root-cause analysis system
built with [LangGraph](https://langchain-ai.github.io/langgraph/).  Four
specialized agents collaborate in an iterative loop:

1. **Diagnostic Agent** — generates hypotheses about potential root causes
2. **Retrieval Agent** — gathers evidence using multiple search strategies
3. **Evaluation Agent** — assesses evidence, detects causal dependencies
4. **Analysis Agent** — synthesizes findings into a root-cause report

The key output is the **Investigation Graph** — a directed acyclic graph (DAG)
that captures every hypothesis, piece of evidence, and causal relationship
discovered during the investigation.  It is not a side-effect; it *is* the
computation state.

## Setup

```bash
pip install -r requirements.txt
cp .env.example .env
# Add your OpenAI API key to .env
```

In [ ]:
# Cell 1: Environment setup
import os
import sys

# Ensure the project root is on the path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from dotenv import load_dotenv
load_dotenv(os.path.join(project_root, ".env"))

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env"
print(f"Using model: {os.environ.get('OPENAI_MODEL', 'gpt-4o-mini')}")

## Run an Investigation

We'll start with the **Kubernetes Disk Pressure** scenario: the checkout
service is unreachable and customers can't complete orders.  The root cause
is a chain of events: disk fills up → kubelet evicts the pod → deployment
has zero replicas → service goes down.

The agents don't know this yet — they will discover it iteratively.

In [ ]:
# Cell 2: Run the investigation pipeline
from src.graph.builder import build_investigation_graph
from src.evidence.mock_data import KUBERNETES_DISK_PRESSURE

app = build_investigation_graph()

initial_state = {
    "problem_statement": KUBERNETES_DISK_PRESSURE["problem_statement"],
    "evidence": KUBERNETES_DISK_PRESSURE["evidence_sources"],
    "hypotheses": [],
    "evaluations": [],
    "nodes": [],
    "edges": [],
    "iteration": 0,
    "max_iterations": 5,
    "converged": False,
    "root_cause": None,
}

print("Starting investigation...")
result = app.invoke(initial_state)
print("Investigation complete.")

In [ ]:
# Cell 3: Inspect the results
print(f"Root cause: {result['root_cause']}")
print(f"Iterations: {result['iteration']}")
print(f"Converged: {result['converged']}")
print(f"Graph nodes: {len(result['nodes'])}")
print(f"Graph edges: {len(result['edges'])}")
print()

# Hypothesis summary
print("Hypothesis outcomes:")
for h in result["hypotheses"]:
    print(f"  [{h.get('status', '?'):>10}] {h['id']}: {h['text'][:80]}")

In [ ]:
# Cell 4: Visualize the Investigation Graph
from src.graph.visualization import plot_investigation_graph

plot_investigation_graph(
    result["nodes"],
    result["edges"],
    save_path="investigation_graph.png",
)

In [ ]:
# Cell 5: Print the full Investigation Graph as JSON
import json
from src.graph.visualization import export_graph_json

graph_data = export_graph_json(result["nodes"], result["edges"])
print(json.dumps(graph_data, indent=2))

## Trace the Reasoning Path

The Investigation Graph captures *every* step of the reasoning process.
We can extract the path from the initial problem to the final root cause
to understand exactly how the agents arrived at their conclusion.

In [ ]:
# Cell 6: Extract and display the root-cause reasoning path
from src.graph.visualization import extract_root_cause_path

path = extract_root_cause_path(result["nodes"], result["edges"])

print("Reasoning path (problem → root cause):")
print()
for i, step in enumerate(path):
    prefix = "  → " if i > 0 else "  ● "
    confidence = step.get('confidence')
    conf_str = f" [{confidence:.0%}]" if confidence is not None else ""
    print(f"{prefix}[{step['type']:>12}] {step['text'][:80]}{conf_str}")

## Run a Different Scenario

The same pipeline handles different kinds of problems.  Let's try the
**Database Connection Pool** scenario — an analytics query holds too
many connections, starving the application.

In [ ]:
# Cell 7: Database Connection Pool scenario
from src.evidence.mock_data import DATABASE_CONNECTION_POOL

db_result = app.invoke({
    "problem_statement": DATABASE_CONNECTION_POOL["problem_statement"],
    "evidence": DATABASE_CONNECTION_POOL["evidence_sources"],
    "hypotheses": [],
    "evaluations": [],
    "nodes": [],
    "edges": [],
    "iteration": 0,
    "max_iterations": 5,
    "converged": False,
    "root_cause": None,
})

print(f"Root cause: {db_result['root_cause']}")
print(f"Iterations: {db_result['iteration']}")
print(f"Graph nodes: {len(db_result['nodes'])}, edges: {len(db_result['edges'])}")

## Compare Investigation Graph Topologies

Different problems produce different graph shapes, but they follow the
same structural pattern: problem → hypotheses → evidence → (pruning) →
root cause.  Compare the node/edge type distributions across scenarios
to see how the agents adapt their investigation strategy.

In [ ]:
# Cell 8: Compare graph topologies
k8s_graph = export_graph_json(result["nodes"], result["edges"])
db_graph = export_graph_json(db_result["nodes"], db_result["edges"])

print("Scenario comparison:")
print(f"{'':>30} {'K8s Disk':>12} {'DB Pool':>12}")
print(f"{'Total nodes':>30} {k8s_graph['summary']['total_nodes']:>12} {db_graph['summary']['total_nodes']:>12}")
print(f"{'Total edges':>30} {k8s_graph['summary']['total_edges']:>12} {db_graph['summary']['total_edges']:>12}")
print()

all_types = sorted(set(list(k8s_graph['summary']['node_types']) + list(db_graph['summary']['node_types'])))
print("Node type distribution:")
for t in all_types:
    k = k8s_graph['summary']['node_types'].get(t, 0)
    d = db_graph['summary']['node_types'].get(t, 0)
    print(f"  {t:>20}: {k:>5}  {d:>5}")